# ZAC 2026 LifeMate — Inference & Timing Notebook

Notebook chuẩn theo Phụ lục (Mục 7) của `Docker_Guide_ZAC2026_LifeMate.pdf`.
Chạy lần lượt 4 bước và xuất kết quả ra `/result/jupyter_submission.jsonl` (kèm trường `time` tính bằng ms).

In [ ]:
# Bước 1: Set seed cố định và cấu hình môi trường offline
import os
import sys
import random
import numpy as np
import torch
from pathlib import Path

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

CODE_DIR = Path("/code") if Path("/code/predict.py").is_file() else Path.cwd()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
from predict import (
    seed_everything as _sync_seeds,
    detect_split,
    load_pipeline_resources,
    build_sample_agent,
    infer_one,
)
_sync_seeds(42)

In [ ]:
# Bước 2: Nạp mô hình (ASR + LLM < 5B) và tài nguyên cần thiết (100% offline)
resources = load_pipeline_resources()
asr_model = resources["transcriber"]
model = resources["llm"]

In [ ]:
# Bước 3: Đọc nội dung các test case từ thư mục đầu vào (/data)
import json

ROOT = Path(os.environ.get("DATA_DIR", "/data"))
SPLIT = detect_split(ROOT, os.environ.get("SPLIT"))
split_dir = ROOT / SPLIT

if str(split_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(split_dir.resolve()))
from zaic_tools import ToolEnv
from zaic_tools.validate import catalog_tools

catalog = json.loads((split_dir / f"{SPLIT}_tools_catalog.json").read_text(encoding="utf-8"))
catalog_tools_map = catalog_tools(catalog)
catalog_apps_map = {app["id"]: app for app in catalog.get("apps", [])}

with open(split_dir / f"{SPLIT}.jsonl", encoding="utf-8") as f:
    test_cases = [json.loads(line) for line in f if line.strip()]

agent = build_sample_agent(
    resources=resources,
    catalog_tools_map=catalog_tools_map,
    catalog_apps_map=catalog_apps_map,
)
print(f"Loaded {len(test_cases)} test cases from split '{SPLIT}'")

In [ ]:
# Bước 4: Thực hiện dự đoán từng mẫu, đo thời gian và ghi /result/jupyter_submission.jsonl
import time

all_result = []
t_total = time.perf_counter()

for item in test_cases:
    t1 = time.perf_counter()
    env = ToolEnv(SPLIT, item["id"], root=ROOT)
    tool_calls, state_hash = infer_one(item, env, asr_model, agent, split_dir)
    ms = int((time.perf_counter() - t1) * 1000)
    all_result.append({
        "id": item["id"],
        "tool_calls": tool_calls,
        "state_hash": state_hash,
        "time": ms,
    })

result_dir = Path("/result")
result_dir.mkdir(parents=True, exist_ok=True)
out_file = result_dir / "jupyter_submission.jsonl"
with open(out_file, "w", encoding="utf-8") as f:
    for r in all_result:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

total_ms = int((time.perf_counter() - t_total) * 1000)
avg_ms = total_ms / max(1, len(all_result))
print(f"predict: {len(all_result)} mẫu, {total_ms} ms, trung bình {avg_ms:.1f} ms/mẫu -> {out_file}")